# Environment

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#your path
%cd /content/drive/MyDrive/personal_project/ViKIS

In [ ]:
!pip install -r requirements.txt # must restart session

Nhớ cd thư mục root lại sau khi khởi động lại phiên


In [ ]:
#your path
%cd /content/drive/MyDrive/personal_project/ViKIS

# Build Database

### Cấu hình chung (cell này dùng để tiện cho sửa đổi cấu hình)

In [ ]:
%%writefile configs/config.yaml

# CẤU HÌNH ĐƯỜNG DẪN HỆ THỐNG

paths:
  raw_videos_dir: "data/raw_videos"
  keyframes_dir: "data/keyframes"
  transcripts_dir: "data/transcripts"
  ocr_cache_dir: "data/ocr_cache"
  logs_dir: "logs"


# CẤU HÌNH PHẦN CỨNG & TIẾN TRÌNH

system:
  device: "cuda"              # "cuda" hoặc "cpu"
  num_workers: 4              # Số tiến trình worker cho Ingestion (Multiprocessing)
  batch_size: 16              # Batch size chung khi xử lý
  random_seed: 42


# CẤU HÌNH TIỀN XỬ LÝ VIDEO & KEYFRAME

ingestion:
  scene_detection:
    adaptive_threshold: 3.0   # Ngưỡng nhạy cảm chuyển cảnh của PySceneDetect
    min_scene_len_sec: 1.0    # Độ dài tối thiểu của 1 shot (giây)

  keyframe_extraction:
    short_shot_threshold: 8.0 # Ngưỡng phân định shot ngắn vs dài (giây)
    sample_rate_short: 0.5    # Khoảng cách lấy mẫu candidate cho shot ngắn (giây)
    sample_rate_long: 2.0     # Khoảng cách lấy mẫu candidate cho shot dài (giây)
    safe_margin_sec: 0.3      # Bỏ qua biên đầu/đuôi shot để tránh nhòe chuyển cảnh
    min_sharpness: 80.0       # Ngưỡng Laplacian Variance tối thiểu
    min_brightness: 25.0      # Ngưỡng độ sáng tối thiểu (Mean Grayscale [0-255])
    phash_threshold: 12       # Khoảng cách Hamming tối thiểu của pHash để giữ frame

  ocr:
    enabled: true
    min_confidence: 0.5       # Độ tin cậy tối thiểu để giữ lại từ ngữ nhận diện


# CẤU HÌNH TRUY XUẤT & TỔNG HỢP (RETRIEVAL)

retrieval:
  temporal_window_margin: 3.0 # Độ trễ cho phép khi so khớp Visual và Audio (giây)
  rrf_k: 60                   # Hằng số chuẩn hóa Reciprocal Rank Fusion
  coarse_to_fine:
    enabled: true
    expansion_window_sec: 1.0 # Mở rộng vùng tìm kiếm để giải mã On-the-fly
    dense_fps: 10.0            # Tốc độ quét frame chi tiết trong RAM

reranker:
  model_id: "BAAI/bge-reranker-v2-m3"
  max_length: 512


### Chạy scripts

In [ ]:
!python -m scripts.run_indexing

### Lỗi không nhận diện được Audio
Nếu chạy qua cloud như google drive có thể bị nghẽn I/O. Chạy cell bên dưới trước rồi indexing sau (khi indexing sẽ tự lấy json)

In [ ]:
import os
import json

from src.ingestion.asr_pipeline import ASRPipeline


asr = ASRPipeline()
raw_dir = "data/raw_videos"

if not os.path.exists(raw_dir):
    print(f"[ASR] Thư mục raw video không tồn tại: {raw_dir}")
    raise SystemExit(1)

video_extensions = (".mp4", ".mkv", ".avi", ".mov", ".webm")
videos = []
for root, _, files in os.walk(raw_dir):
    for file in files:
        if file.lower().endswith(video_extensions):
            videos.append(os.path.join(root, file))

videos = sorted(set(videos))
if not videos:
    print(f"[ASR] Không tìm thấy video nào trong {raw_dir}")
    raise SystemExit(0)

print(f"[ASR] Tìm thấy {len(videos)} video. Bắt đầu transcribe...\n")

for idx, video_path in enumerate(videos, start=1):
    video_name = os.path.basename(video_path)
    print(f"\n=== [{idx}/{len(videos)}] {video_name} ===")
    transcripts = asr.transcribe(video_path, use_cache=False)

    if not transcripts:
        print("[ASR] Không phát hiện đoạn thoại nào.")
        continue

Nếu truy vấn là các câu hỏi ngắn (tối đa 64 token) có thể thay mô hình `jinaai/jina-clip-v2` bằng `google/siglip2-so400m-patch14-384`

In [ ]:
!python -m scripts.run_indexing

# UI: dùng ngrok


In [ ]:
!pip install pyngrok -q

thêm khoá bí mật `NGROK_TOKEN` rồi chạy cell bên dưới sau đó nhấn vào link hiện ở kết quả để vào giao diện streamlit.

In [ ]:
from pyngrok import ngrok
import os
import sys
from google.colab import userdata

# 1. Lấy Authtoken từ Colab Secrets
try:
    token = userdata.get('NGROK_TOKEN')
    ngrok.set_auth_token(token)
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    print("Vui lòng thêm secret 'NGROK_TOKEN' vào mục Secrets (biểu tượng chìa khoá) của Colab.")
    sys.exit("Cấu hình thiếu authtoken.")

# 2. Xóa các tunnel cũ nếu có
ngrok.kill()

# 3. Mở tunnel tới port 8501
try:
    public_url = ngrok.connect(8501)
    print(f"Giao diện Streamlit của bạn đã mở tại: {public_url.public_url}")

    # 4. Khởi chạy Streamlit (Chạy trực tiếp trong cell này)
    os.system("streamlit run app/streamlit_app.py --server.enableCORS false --server.enableXsrfProtection false")
except Exception as e:
    print(f"Không thể tạo tunnel: {e}")